In [1]:
from pathlib import Path
import sys

import polars as pl

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

from ufc_scraper.utils import read_totals


In [2]:
totals = read_totals().collect()

In [ ]:
totals.head()

In [3]:
import polars as pl


def standardize_dataframe(
    df: pl.DataFrame,
    empty_to_null: bool = False,
) -> pl.DataFrame:
    """
    Standardize a Polars DataFrame.

    - Strips whitespace from column names.
    - Strips whitespace from all string values.
    - Removes surrounding double quotes from string values.
    - Optionally converts empty strings to nulls.
    """

    # Clean column names
    df = df.rename(
        {
            col: col.strip()
            for col in df.columns
        }
    )

    string_cols = [
        col
        for col, dtype in df.schema.items()
        if dtype == pl.String
    ]

    exprs = []

    for col in string_cols:
        expr = (
            pl.col(col)
            .str.strip_chars()
            .str.strip_chars('"')
        )

        if empty_to_null:
            expr = (
                pl.when(expr == "")
                .then(None)
                .otherwise(expr)
            )

        exprs.append(expr.alias(col))

    return df.with_columns(*exprs)

In [4]:
TOTALS_VALUE_CATEGORIES = ["sig_str", "total_str", "td"]

TOTALS_PCT_CATEGORIES = ["sig_str_pct", "td_pct", "total_str_pct"]

TOTALS_INTEGER_CATEGORIES = ["kd", "sub_att", "rev"]

TOTALS_METRICS = [
    "landed",
    "attempted",
    "pct",
    "kd",
    "sub_att",
    "rev",
    "ctrl_seconds",
]

FIGHTERS = [1, 2]

In [5]:
def transform_totals(totals: pl.DataFrame) -> pl.DataFrame:

    exprs_values = []
    exprs_pct = []
    exprs_int = []
    exprs_values_pct = []

    cols_to_drop = (
        [
            f"{category}_fighter{fighter}"
            for category in TOTALS_VALUE_CATEGORIES
            for fighter in FIGHTERS
        ]

        + [
            f"ctrl_fighter{fighter}"
            for fighter in FIGHTERS
        ]
    )
    

    for category in TOTALS_VALUE_CATEGORIES:
        for fighter in FIGHTERS:
            exprs_values_pct.append(
                (
                    pl.when(
                        pl.col(f"{category}_attempted_fighter{fighter}") > 0
                    )
                    .then(
                        100
                        * pl.col(f"{category}_landed_fighter{fighter}")
                        / pl.col(f"{category}_attempted_fighter{fighter}")
                    )
                    .otherwise(0)
                    .round(2)
                    .alias(f"{category}_pct_fighter{fighter}")
                )
            )

    # "X of Y" columns
    for category in TOTALS_VALUE_CATEGORIES:
        for fighter in FIGHTERS:

            exprs_values.extend([
                pl.col(f"{category}_fighter{fighter}")
                .str.extract(r"(\d+) of (\d+)", 1)
                .cast(pl.Int64)
                .alias(f"{category}_landed_fighter{fighter}"),

                pl.col(f"{category}_fighter{fighter}")
                .str.extract(r"(\d+) of (\d+)", 2)
                .cast(pl.Int64)
                .alias(f"{category}_attempted_fighter{fighter}")
            ])

    for category in TOTALS_PCT_CATEGORIES:
        for fighter in FIGHTERS:

            exprs_pct.append(
                pl.col(f"{category}_fighter{fighter}")
                .str.replace("%", "")
                .cast(pl.Float64, strict=False)
                .fill_null(0)
                .alias(f"{category}_fighter{fighter}")
            )

    # integer columns
    for category in TOTALS_INTEGER_CATEGORIES:
        for fighter in FIGHTERS:

            exprs_int.append(
                pl.col(f"{category}_fighter{fighter}")
                .cast(pl.Int64)
                .alias(f"{category}_fighter{fighter}")
            )

    return (
        totals
        .drop("sig_str_pct_fighter1","sig_str_pct_fighter2"
              ,"td_pct_fighter1","td_pct_fighter2")
        .with_columns(*exprs_values)
        .with_columns(*exprs_int)
        .with_columns(*exprs_values_pct)
        .with_columns(

            # round
            pl.col("round_label")
            .str.extract(r"(\d+)")
            .cast(pl.Int64)
            .alias("round"),

            # control time -> seconds
            (
                pl.col("ctrl_fighter1")
                .str.extract(r"(\d+):(\d+)", 1)
                .cast(pl.Int64)
                * 60
                +
                pl.col("ctrl_fighter1")
                .str.extract(r"(\d+):(\d+)", 2)
                .cast(pl.Int64)
            ).alias("ctrl_seconds_fighter1"),

            (
                pl.col("ctrl_fighter2")
                .str.extract(r"(\d+):(\d+)", 1)
                .cast(pl.Int64)
                * 60
                +
                pl.col("ctrl_fighter2")
                .str.extract(r"(\d+):(\d+)", 2)
                .cast(pl.Int64)
            ).alias("ctrl_seconds_fighter2"),

            pl.col("scraped_at")
            .str.to_datetime(format="%Y-%m-%dT%H:%M:%S%z"),

            pl.col("ingested_at")
            .str.to_datetime(format="%Y-%m-%dT%H:%M:%S%z"),

            pl.col("fighter1").str.strip_chars(),
            pl.col("fighter2").str.strip_chars(),
        )
        .drop("round_label")
        .drop(cols_to_drop)
    )

In [6]:
def build_totals_fighter_view(
    df: pl.DataFrame,
    fighter_num: int,
) -> pl.DataFrame:
    
    BASE_COLS = ["fight_id", "event_id", "round", "scraped_at", "ingested_at",]
    
    other = 2 if fighter_num == 1 else 1

    exprs = [
        pl.col(f"fighter{fighter_num}").alias("fighter"),
        pl.col(f"fighter{other}").alias("opponent"),
    ]

    for category in TOTALS_VALUE_CATEGORIES:
        exprs.extend([
            pl.col(f"{category}_landed_fighter{fighter_num}")
            .alias(f"{category}_landed"),

            pl.col(f"{category}_attempted_fighter{fighter_num}")
            .alias(f"{category}_attempted"),
        ])

    for category in TOTALS_PCT_CATEGORIES:
        exprs.append(
            pl.col(f"{category}_fighter{fighter_num}")
            .alias(category)
        )

    for category in TOTALS_INTEGER_CATEGORIES:
        exprs.append(
            pl.col(f"{category}_fighter{fighter_num}")
            .alias(category)
        )

    exprs.append(
        pl.col(f"ctrl_seconds_fighter{fighter_num}")
        .alias("ctrl_seconds")
    )

    return df.select(*BASE_COLS, *exprs)

In [7]:
def create_totals_long(
    totals: pl.DataFrame,
) -> pl.DataFrame:

    fighter1_df = build_totals_fighter_view(totals, 1)
    fighter2_df = build_totals_fighter_view(totals, 2)

    return pl.concat(
        [fighter1_df, fighter2_df],
        how="vertical",
    )

In [8]:
totals_long = (
    totals
    .pipe(standardize_dataframe, empty_to_null=True)
    .pipe(transform_totals)
    .pipe(create_totals_long)
)

In [9]:
totals_long.shape

(52, 20)

In [10]:
totals_long.head(10)

fight_id,event_id,round,scraped_at,ingested_at,fighter,opponent,sig_str_landed,sig_str_attempted,total_str_landed,total_str_attempted,td_landed,td_attempted,sig_str_pct,td_pct,total_str_pct,kd,sub_att,rev,ctrl_seconds
str,str,i64,"datetime[μs, UTC]","datetime[μs, UTC]",str,str,i64,i64,i64,i64,i64,i64,f64,f64,f64,i64,i64,i64,i64
"""fb77b08a90b92d5b""","""a0a69dc9914ef6e1""",1,2026-09-03 12:54:37 UTC,2026-09-03 13:23:47 UTC,"""Anthony Hernandez""","""Gregory Rodrigues""",10,19,10,19,0,7,52.63,0.0,52.63,0,0,0,168
"""fb77b08a90b92d5b""","""a0a69dc9914ef6e1""",2,2026-09-03 12:54:37 UTC,2026-09-03 13:23:47 UTC,"""Anthony Hernandez""","""Gregory Rodrigues""",19,39,19,39,1,2,48.72,50.0,48.72,0,0,0,62
"""fb77b08a90b92d5b""","""a0a69dc9914ef6e1""",3,2026-09-03 12:54:37 UTC,2026-09-03 13:23:47 UTC,"""Anthony Hernandez""","""Gregory Rodrigues""",12,20,17,26,1,9,60.0,11.11,65.38,0,0,0,229
"""fb77b08a90b92d5b""","""a0a69dc9914ef6e1""",4,2026-09-03 12:54:37 UTC,2026-09-03 13:23:47 UTC,"""Anthony Hernandez""","""Gregory Rodrigues""",25,50,26,51,1,6,50.0,16.67,50.98,0,0,0,80
"""fb77b08a90b92d5b""","""a0a69dc9914ef6e1""",5,2026-09-03 12:54:37 UTC,2026-09-03 13:23:47 UTC,"""Anthony Hernandez""","""Gregory Rodrigues""",24,31,39,49,0,5,77.42,0.0,79.59,0,0,0,101
"""a57ccf31c5f6f82b""","""a0a69dc9914ef6e1""",1,2026-09-03 12:54:49 UTC,2026-09-03 13:23:47 UTC,"""Serghei Spivac""","""Vitor Petrino""",7,29,7,29,0,0,24.14,0.0,24.14,0,0,0,0
"""a57ccf31c5f6f82b""","""a0a69dc9914ef6e1""",2,2026-09-03 12:54:49 UTC,2026-09-03 13:23:47 UTC,"""Serghei Spivac""","""Vitor Petrino""",3,5,12,14,0,1,60.0,0.0,85.71,0,0,1,22
"""a57ccf31c5f6f82b""","""a0a69dc9914ef6e1""",3,2026-09-03 12:54:49 UTC,2026-09-03 13:23:47 UTC,"""Serghei Spivac""","""Vitor Petrino""",13,33,14,34,0,1,39.39,0.0,41.18,0,0,0,0
"""0eba299c9b6760fa""","""a0a69dc9914ef6e1""",1,2026-09-03 12:55:00 UTC,2026-09-03 13:23:47 UTC,"""Reinier de Ridder""","""Roman Dolidze""",71,83,106,119,0,0,85.54,0.0,89.08,0,0,0,221
